# LoRA (Low-Rank Adaptation) Fine-Tuning Demonstration

This notebook demonstrates the complete workflow of fine-tuning a language model using LoRA technique. We'll cover:

1. **Model Setup**: Loading a pre-trained transformer model
2. **LoRA Implementation**: Adding LoRA adapters to the model
3. **Dataset Preparation**: Preparing training data
4. **Training Process**: Fine-tuning with LoRA
5. **Evaluation**: Comparing metrics before and after fine-tuning
6. **Analysis**: Understanding LoRA's efficiency benefits

## What is LoRA?

LoRA (Low-Rank Adaptation) is an efficient fine-tuning technique that:
- Freezes the original model weights
- Adds trainable low-rank decomposition matrices
- Significantly reduces the number of trainable parameters
- Maintains comparable performance to full fine-tuning

In [ ]:
# Install required packages
!pip install transformers datasets peft torch accelerate evaluate matplotlib seaborn pandas numpy tqdm

In [ ]:
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset, Dataset
import evaluate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import json
import time
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Model Setup

We'll start with a pre-trained BERT model for sequence classification. This will serve as our base model before applying LoRA.

In [ ]:
# Model configuration
MODEL_NAME = "bert-base-uncased"
NUM_LABELS = 2  # Binary classification

# Load tokenizer and model
print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=NUM_LABELS,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 2. Dataset Preparation

We'll use a sentiment analysis dataset to demonstrate the fine-tuning process.

In [ ]:
# Load and prepare dataset
print("Loading dataset...")
dataset = load_dataset('imdb', split='train[:5000]')  # Using subset for demonstration
dataset = dataset.train_test_split(test_size=0.2, seed=42)

print(f"Training samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['test'])}")

# Sample data preview
print("\nSample data:")
for i in range(2):
    print(f"Text: {dataset['train'][i]['text'][:100]}...")
    print(f"Label: {dataset['train'][i]['label']}\n")

In [ ]:
# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding=False,  # We'll pad dynamically
        max_length=512
    )

# Tokenize datasets
print("Tokenizing datasets...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Tokenization complete!")

# Data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 3. LoRA Configuration and Setup

Now we'll configure LoRA parameters and apply them to our model. The key parameters are:
- `r`: The rank of the low-rank matrices (lower = fewer parameters)
- `alpha`: LoRA scaling parameter
- `dropout`: Dropout probability for LoRA layers

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,  # Sequence classification
    r=16,  # Rank of adaptation
    lora_alpha=32,  # LoRA scaling parameter
    lora_dropout=0.1,  # LoRA dropout
    target_modules=["query", "value"],  # Target attention modules
    bias="none",
)

print("LoRA Configuration:")
print(f"Rank (r): {lora_config.r}")
print(f"Alpha: {lora_config.lora_alpha}")
print(f"Dropout: {lora_config.lora_dropout}")
print(f"Target modules: {lora_config.target_modules}")

In [ ]:
# Apply LoRA to the model
print("Applying LoRA to the model...")
lora_model = get_peft_model(model, lora_config)

# Print trainable parameters comparison
def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"Trainable params: {trainable_params:,} || "
        f"All params: {all_param:,} || "
        f"Trainable%: {100 * trainable_params / all_param:.4f}%"
    )

print("\nParameter comparison:")
print_trainable_parameters(lora_model)

# Move model to device
lora_model.to(device)

## 4. Training Setup and Evaluation Metrics

We'll set up the training arguments and evaluation metrics to track the model's performance.

In [ ]:
# Load evaluation metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    precision = precision_metric.compute(predictions=predictions, references=labels, average="weighted")["precision"]
    recall = recall_metric.compute(predictions=predictions, references=labels, average="weighted")["recall"]
    
    return {
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./lora-sentiment-results",
    learning_rate=2e-4,  # Higher learning rate for LoRA
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    warmup_steps=100,
    fp16=torch.cuda.is_available(),  # Mixed precision training if CUDA available
    dataloader_pin_memory=False,
    remove_unused_columns=False,
)

print("Training Arguments:")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"FP16: {training_args.fp16}")

## 5. Baseline Evaluation

Before training, let's evaluate the model's performance on our validation set to establish a baseline.

In [ ]:
# Create trainer
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Baseline evaluation
print("Evaluating baseline performance...")
baseline_metrics = trainer.evaluate()

print("\nBaseline Performance:")
for key, value in baseline_metrics.items():
    if key.startswith('eval_'):
        metric_name = key.replace('eval_', '').replace('_', ' ').title()
        print(f"{metric_name}: {value:.4f}")

# Store baseline for comparison
baseline_results = {
    'accuracy': baseline_metrics['eval_accuracy'],
    'f1': baseline_metrics['eval_f1'],
    'precision': baseline_metrics['eval_precision'],
    'recall': baseline_metrics['eval_recall']
}

## 6. LoRA Fine-Tuning

Now let's train the model with LoRA adapters. We'll track training metrics and time.

In [ ]:
# Start training
print("Starting LoRA fine-tuning...")
start_time = time.time()

# Train the model
train_result = trainer.train()

training_time = time.time() - start_time
print(f"\nTraining completed in {training_time:.2f} seconds ({training_time/60:.2f} minutes)")

# Training statistics
print("\nTraining Statistics:")
print(f"Total training steps: {train_result.global_step}")
print(f"Final training loss: {train_result.training_loss:.4f}")

## 7. Post-Training Evaluation

Let's evaluate the fine-tuned model and compare it with the baseline.

In [ ]:
# Post-training evaluation
print("Evaluating fine-tuned model...")
final_metrics = trainer.evaluate()

print("\nFine-tuned Performance:")
for key, value in final_metrics.items():
    if key.startswith('eval_'):
        metric_name = key.replace('eval_', '').replace('_', ' ').title()
        print(f"{metric_name}: {value:.4f}")

# Store final results
final_results = {
    'accuracy': final_metrics['eval_accuracy'],
    'f1': final_metrics['eval_f1'],
    'precision': final_metrics['eval_precision'],
    'recall': final_metrics['eval_recall']
}

## 8. Performance Comparison and Analysis

Let's visualize the improvements achieved through LoRA fine-tuning.

In [ ]:
# Create comparison DataFrame
comparison_df = pd.DataFrame({
    'Baseline': [baseline_results[metric] for metric in ['accuracy', 'f1', 'precision', 'recall']],
    'LoRA Fine-tuned': [final_results[metric] for metric in ['accuracy', 'f1', 'precision', 'recall']]
}, index=['Accuracy', 'F1 Score', 'Precision', 'Recall'])

# Calculate improvements
comparison_df['Improvement'] = comparison_df['LoRA Fine-tuned'] - comparison_df['Baseline']
comparison_df['Improvement (%)'] = (comparison_df['Improvement'] / comparison_df['Baseline']) * 100

print("Performance Comparison:")
print(comparison_df.round(4))

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Performance comparison bar chart
comparison_df[['Baseline', 'LoRA Fine-tuned']].plot(kind='bar', ax=ax1, 
                                                     color=['lightcoral', 'lightblue'])
ax1.set_title('Performance Comparison: Baseline vs LoRA Fine-tuned', fontsize=14, fontweight='bold')
ax1.set_ylabel('Score')
ax1.set_xlabel('Metrics')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45)

# Improvement percentage chart
colors = ['green' if x > 0 else 'red' for x in comparison_df['Improvement (%)']]
comparison_df['Improvement (%)'].plot(kind='bar', ax=ax2, color=colors, alpha=0.7)
ax2.set_title('Relative Improvement (%)', fontsize=14, fontweight='bold')
ax2.set_ylabel('Improvement (%)')
ax2.set_xlabel('Metrics')
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.5)
ax2.grid(True, alpha=0.3)
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

## 9. Training History Analysis

Let's examine the training history to understand how the model improved over time.

In [ ]:
# Extract training history
log_history = trainer.state.log_history

# Separate training and evaluation logs
train_logs = [log for log in log_history if 'loss' in log and 'eval_loss' not in log]
eval_logs = [log for log in log_history if 'eval_loss' in log]

if train_logs and eval_logs:
    # Create training history plot
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    # Training loss
    train_steps = [log['step'] for log in train_logs]
    train_losses = [log['loss'] for log in train_logs]
    ax1.plot(train_steps, train_losses, 'b-', linewidth=2, label='Training Loss')
    ax1.set_title('Training Loss Over Time', fontweight='bold')
    ax1.set_xlabel('Steps')
    ax1.set_ylabel('Loss')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Evaluation metrics
    eval_steps = [log['step'] for log in eval_logs]
    eval_losses = [log['eval_loss'] for log in eval_logs]
    eval_accuracies = [log['eval_accuracy'] for log in eval_logs]
    eval_f1s = [log['eval_f1'] for log in eval_logs]
    
    ax2.plot(eval_steps, eval_losses, 'r-', linewidth=2, label='Validation Loss')
    ax2.set_title('Validation Loss Over Time', fontweight='bold')
    ax2.set_xlabel('Steps')
    ax2.set_ylabel('Loss')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    ax3.plot(eval_steps, eval_accuracies, 'g-', linewidth=2, label='Accuracy')
    ax3.set_title('Validation Accuracy Over Time', fontweight='bold')
    ax3.set_xlabel('Steps')
    ax3.set_ylabel('Accuracy')
    ax3.grid(True, alpha=0.3)
    ax3.legend()
    
    ax4.plot(eval_steps, eval_f1s, 'm-', linewidth=2, label='F1 Score')
    ax4.set_title('Validation F1 Score Over Time', fontweight='bold')
    ax4.set_xlabel('Steps')
    ax4.set_ylabel('F1 Score')
    ax4.grid(True, alpha=0.3)
    ax4.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"Training progressed over {len(train_logs)} training steps")
    print(f"Best validation accuracy: {max(eval_accuracies):.4f}")
    print(f"Best validation F1 score: {max(eval_f1s):.4f}")
else:
    print("Training history not available for plotting.")

## 10. LoRA Efficiency Analysis

Let's analyze the efficiency benefits of using LoRA compared to full fine-tuning.

In [ ]:
# Parameter efficiency analysis
def analyze_model_efficiency(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params
    
    return {
        'total': total_params,
        'trainable': trainable_params,
        'frozen': frozen_params,
        'trainable_percentage': (trainable_params / total_params) * 100
    }

# Analyze our LoRA model
lora_efficiency = analyze_model_efficiency(lora_model)

print("LoRA Efficiency Analysis:")
print("="*50)
print(f"Total parameters: {lora_efficiency['total']:,}")
print(f"Trainable parameters: {lora_efficiency['trainable']:,}")
print(f"Frozen parameters: {lora_efficiency['frozen']:,}")
print(f"Trainable percentage: {lora_efficiency['trainable_percentage']:.2f}%")
print(f"Parameter reduction: {100 - lora_efficiency['trainable_percentage']:.2f}%")

# Memory and storage benefits
full_model_size_mb = (lora_efficiency['total'] * 4) / (1024**2)  # Assuming float32
lora_adapter_size_mb = (lora_efficiency['trainable'] * 4) / (1024**2)

print(f"\nStorage Analysis:")
print(f"Full model size (est.): {full_model_size_mb:.2f} MB")
print(f"LoRA adapter size: {lora_adapter_size_mb:.2f} MB")
print(f"Storage reduction: {((full_model_size_mb - lora_adapter_size_mb) / full_model_size_mb) * 100:.2f}%")

In [ ]:
# Visualization of parameter efficiency
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Parameter distribution pie chart
labels = ['Frozen Parameters', 'Trainable Parameters']
sizes = [lora_efficiency['frozen'], lora_efficiency['trainable']]
colors = ['lightcoral', 'lightblue']
explode = (0, 0.1)  # explode the trainable slice

ax1.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.2f%%',
        shadow=True, startangle=90)
ax1.set_title('Parameter Distribution in LoRA Model', fontweight='bold')

# Comparison bar chart
categories = ['Full Fine-tuning', 'LoRA']
trainable_params = [lora_efficiency['total'], lora_efficiency['trainable']]

bars = ax2.bar(categories, trainable_params, color=['red', 'blue'], alpha=0.7)
ax2.set_title('Trainable Parameters Comparison', fontweight='bold')
ax2.set_ylabel('Number of Parameters')
ax2.grid(True, alpha=0.3)

# Add value labels on bars
for bar, value in zip(bars, trainable_params):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{value:,}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 11. Model Inference Testing

Let's test the fine-tuned model with some sample inputs to see how it performs in practice.

In [ ]:
# Sample texts for inference
test_texts = [
    "This movie was absolutely fantastic! The acting was superb and the plot was engaging.",
    "I really didn't enjoy this film. The story was boring and predictable.",
    "The cinematography was beautiful, but the dialogue felt forced and unnatural.",
    "An amazing experience from start to finish. Highly recommended!",
    "Waste of time. Poor acting and terrible script."
]

# Function to make predictions
def predict_sentiment(texts, model, tokenizer):
    model.eval()
    predictions = []
    probabilities = []
    
    with torch.no_grad():
        for text in texts:
            # Tokenize input
            inputs = tokenizer(text, return_tensors="pt", truncation=True, 
                             padding=True, max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # Get prediction
            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1)
            
            pred = torch.argmax(logits, dim=-1).item()
            prob = probs[0][pred].item()
            
            predictions.append(pred)
            probabilities.append(prob)
    
    return predictions, probabilities

# Make predictions
predictions, probabilities = predict_sentiment(test_texts, lora_model, tokenizer)

# Display results
print("Sentiment Analysis Results:")
print("="*80)
label_map = {0: "Negative", 1: "Positive"}

for i, (text, pred, prob) in enumerate(zip(test_texts, predictions, probabilities)):
    sentiment = label_map[pred]
    print(f"Text {i+1}: {text[:60]}{'...' if len(text) > 60 else ''}")
    print(f"Prediction: {sentiment} (Confidence: {prob:.3f})")
    print("-" * 80)

## 12. Summary and Key Takeaways

Let's summarize our findings and the benefits of using LoRA for fine-tuning.

In [ ]:
# Create comprehensive summary
summary_data = {
    'Metric': ['Training Time (minutes)', 'Trainable Parameters', 'Parameter Reduction (%)', 
               'Accuracy Improvement', 'F1 Score Improvement', 'Storage Savings (MB)'],
    'Value': [
        f"{training_time/60:.2f}",
        f"{lora_efficiency['trainable']:,}",
        f"{100 - lora_efficiency['trainable_percentage']:.2f}%",
        f"{comparison_df.loc['Accuracy', 'Improvement']:.4f}",
        f"{comparison_df.loc['F1 Score', 'Improvement']:.4f}",
        f"{full_model_size_mb - lora_adapter_size_mb:.2f}"
    ]
}

summary_df = pd.DataFrame(summary_data)

print("\n" + "="*60)
print("LORA FINE-TUNING SUMMARY")
print("="*60)
print(summary_df.to_string(index=False))
print("="*60)

print("\nKey Benefits of LoRA:")
print("• Dramatic reduction in trainable parameters (>99%)")
print("• Maintains model performance with minimal parameter updates")
print("• Faster training and reduced memory requirements")
print("• Easy to deploy and share (small adapter files)")
print("• Can be combined or switched for different tasks")
print("• Preserves original model knowledge while adapting to new tasks")

print("\nLoRA Configuration Used:")
print(f"• Rank (r): {lora_config.r}")
print(f"• Alpha: {lora_config.lora_alpha}")
print(f"• Dropout: {lora_config.lora_dropout}")
print(f"• Target modules: {', '.join(lora_config.target_modules)}")

In [ ]:
# Save results for future reference
results_summary = {
    'model_name': MODEL_NAME,
    'lora_config': {
        'r': lora_config.r,
        'alpha': lora_config.lora_alpha,
        'dropout': lora_config.lora_dropout,
        'target_modules': lora_config.target_modules
    },
    'baseline_metrics': baseline_results,
    'final_metrics': final_results,
    'efficiency': lora_efficiency,
    'training_time': training_time
}

# Save to JSON file
with open('lora_experiment_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("Results saved to 'lora_experiment_results.json'")
print("\nExperiment completed successfully! 🎉")

## Conclusion

This notebook demonstrated the complete workflow of using LoRA for efficient fine-tuning of language models. We successfully:

1. **Loaded a pre-trained model** and established baseline performance
2. **Applied LoRA adapters** with minimal trainable parameters
3. **Fine-tuned the model** on a sentiment analysis task
4. **Achieved performance improvements** while using <1% of the original parameters
5. **Analyzed the efficiency gains** in terms of parameters, memory, and storage

LoRA proves to be an excellent technique for practical fine-tuning scenarios where computational resources are limited but high performance is still required. The technique is particularly valuable for:

- **Resource-constrained environments**
- **Multi-task scenarios** where different adapters can be swapped
- **Rapid prototyping** of task-specific models
- **Deployment scenarios** where model size matters

The combination of efficiency and effectiveness makes LoRA an essential tool in the modern machine learning toolkit.